# Visualize temporal road predictions

This notebook displays, for selected patch names, the original image, predicted mask, and overlay across all years. If no patch names are provided, it randomly samples patch names.

In [ ]:
"""Visualize multi-year images and predicted road masks.

This script/notebook-style file supports two modes:
1. Provide a list of photo names/stems.
2. If the list is empty, randomly select N photo names.

For each selected photo name, it displays for every year:
    image | predicted mask | overlay
"""

from __future__ import annotations

from pathlib import Path
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

In [ ]:
# =========================================================
# CONFIGURATION

In [ ]:
# =========================================================
OLD_YEARS = [1947, 1957, 1964, 1977, 1991]
RECENT_YEARS = [
    1997, 2006, 2009, 2010, 2011,
    2014, 2015, 2016, 2017, 2019,
    2020, 2021, 2022,
]
YEARS_TO_DISPLAY = OLD_YEARS + RECENT_YEARS

DATASET_NAME = "data_512_no_stride"
PREDICTION_NAME = "final_model_test1"

# Manual mode: provide stems or filenames here.
PHOTO_NAMES = []

# Random mode: used when PHOTO_NAMES is empty.
N_RANDOM_PHOTOS = 5
RANDOM_SEED = 42
REQUIRE_PHOTO_IN_ALL_YEARS = True

PREDICTION_FOLDER_CANDIDATES = ["masks_pred"]
IMAGE_FOLDER_CANDIDATES = ["images"]
OVERLAY_ALPHA = 0.45

In [ ]:
# =========================================================
# PATHS

In [ ]:
# =========================================================
current_dir = Path.cwd().resolve()
project_dir = current_dir.parent if current_dir.name == "notebooks" else current_dir

dataset_dir = project_dir / "data" / "03.pre_ML" / DATASET_NAME
prediction_root_dir = project_dir / "data" / "05.predictions" / PREDICTION_NAME

print("Project directory:   ", project_dir)
print("Dataset directory:   ", dataset_dir)
print("Prediction directory:", prediction_root_dir)

In [ ]:
# =========================================================
# HELPERS

In [ ]:
# =========================================================
def list_files(folder: Path, extensions: list[str]) -> list[Path]:
    if not folder.exists():
        return []
    return sorted(path for path in folder.iterdir() if path.is_file() and path.suffix.lower() in extensions)


def get_available_image_files(year: int) -> dict[str, Path]:
    year_dir = dataset_dir / str(year)
    image_extensions = [".jpg", ".jpeg", ".png", ".tif", ".tiff"]
    files_by_stem: dict[str, Path] = {}

    for folder_name in IMAGE_FOLDER_CANDIDATES:
        image_dir = year_dir / folder_name
        for path in list_files(image_dir, image_extensions):
            if path.stem not in files_by_stem:
                files_by_stem[path.stem] = path
    return files_by_stem


def get_available_prediction_files(year: int) -> dict[str, Path]:
    year_pred_dir = prediction_root_dir / str(year)
    files_by_stem: dict[str, Path] = {}

    for folder_name in PREDICTION_FOLDER_CANDIDATES:
        pred_dir = year_pred_dir / folder_name
        for path in list_files(pred_dir, [".png"]):
            if path.stem not in files_by_stem:
                files_by_stem[path.stem] = path
    return files_by_stem


def build_year_index(years: list[int]) -> dict[int, dict]:
    index: dict[int, dict] = {}
    for year in years:
        image_files = get_available_image_files(year)
        prediction_files = get_available_prediction_files(year)
        matched_stems = sorted(set(image_files) & set(prediction_files))
        index[year] = {"images": image_files, "predictions": prediction_files, "matched_stems": matched_stems}
        print(f"{year}: images={len(image_files):6d} | predictions={len(prediction_files):6d} | matched={len(matched_stems):6d}")
    return index


def normalize_photo_name(name: str) -> str:
    return Path(name).stem


def choose_random_photo_names(index: dict[int, dict], n: int, require_in_all_years: bool = True, seed: int = 42) -> list[str]:
    rng = random.Random(seed)

    if require_in_all_years:
        common_stems: set[str] | None = None
        for year_data in index.values():
            stems = set(year_data["matched_stems"])
            common_stems = stems if common_stems is None else common_stems & stems
        common = sorted(common_stems or [])
        if not common:
            print("No common photo names found across all selected years.")
            return []
        return rng.sample(common, min(n, len(common)))

    all_stems: set[str] = set()
    for year_data in index.values():
        all_stems.update(year_data["matched_stems"])
    stems = sorted(all_stems)
    if not stems:
        print("No matched image/prediction pairs found.")
        return []
    return rng.sample(stems, min(n, len(stems)))


def load_rgb_image(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("RGB"))


def load_binary_mask(path: Path) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Could not read mask: {path}")
    return mask > 0


def make_overlay(image: np.ndarray, mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    if mask.shape[:2] != image.shape[:2]:
        mask = cv2.resize(mask.astype(np.uint8), (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
    overlay = image.astype(np.float32).copy()
    overlay[mask] = (1.0 - alpha) * overlay[mask] + alpha * np.array([255, 0, 0], dtype=np.float32)
    return np.clip(overlay, 0, 255).astype(np.uint8)

In [ ]:
# =========================================================
# DISPLAY FUNCTIONS

In [ ]:
# =========================================================
def display_photo_across_years(photo_name: str, years: list[int], index: dict[int, dict]) -> None:
    stem = normalize_photo_name(photo_name)
    n_years = len(years)

    fig, axes = plt.subplots(n_years, 3, figsize=(12, 3.5 * n_years))
    if n_years == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, year in enumerate(years):
        image_path = index[year]["images"].get(stem)
        mask_path = index[year]["predictions"].get(stem)

        if image_path is None or mask_path is None:
            for col in range(3):
                axes[row, col].axis("off")
            axes[row, 0].set_title(f"{year} - missing")
            axes[row, 0].text(0.5, 0.5, f"Missing image or prediction\n{stem}", ha="center", va="center", fontsize=10)
            continue

        image = load_rgb_image(image_path)
        mask = load_binary_mask(mask_path)
        overlay = make_overlay(image, mask, alpha=OVERLAY_ALPHA)

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f"{year} - image")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask, cmap="gray")
        axes[row, 1].set_title(f"{year} - predicted mask")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(f"{year} - overlay")
        axes[row, 2].axis("off")

    plt.suptitle(f"Photo name: {stem}", fontsize=14)
    plt.tight_layout()
    plt.show()


def display_selected_photos(photo_names: list[str], years: list[int], index: dict[int, dict]) -> None:
    if not photo_names:
        print("No photo names to display.")
        return
    print("Selected photo names:")
    for name in photo_names:
        print(" -", normalize_photo_name(name))
    for photo_name in photo_names:
        display_photo_across_years(photo_name, years, index)

In [ ]:
# =========================================================
# RUN

In [ ]:
# =========================================================
year_index = build_year_index(YEARS_TO_DISPLAY)

if PHOTO_NAMES:
    selected_photo_names = [normalize_photo_name(name) for name in PHOTO_NAMES]
else:
    selected_photo_names = choose_random_photo_names(
        index=year_index,
        n=N_RANDOM_PHOTOS,
        require_in_all_years=REQUIRE_PHOTO_IN_ALL_YEARS,
        seed=RANDOM_SEED,
    )

display_selected_photos(selected_photo_names, YEARS_TO_DISPLAY, year_index)